In [2]:
from pathlib import Path
import sys
import pandas as pd

# PROJECT_ROOT = Path('/Users/choedasom/lab_middle_project')
# assert (PROJECT_ROOT / 'src' / 'feature.py').is_file(), f'경로 확인 필요: {PROJECT_ROOT}'
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

for m in list(sys.modules):
    if m == 'src' or m.startswith('src.'):
        del sys.modules[m]

from src.feature import add_features

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

final_df = pd.read_parquet(PROCESSED_DIR / 'final_df.parquet')
coverage_df = pd.read_csv(PROCESSED_DIR / 'coverage_df.csv', parse_dates=['actual_start_date', 'actual_end_date'])
final_missing_df = pd.read_csv(PROCESSED_DIR / 'final_missing_df.csv')
sp500_universe = pd.read_csv(RAW_DIR / 'sp500_universe.csv')
sp500_beta_df = pd.read_parquet(RAW_DIR / 'sp500_beta_df.parquet')
print(f'final_df: {final_df.shape}, {final_df["Date"].min()} ~ {final_df["Date"].max()}')
print(f'sp500_beta_df: {sp500_beta_df.shape}, {sp500_beta_df["Date"].min()} ~ {sp500_beta_df["Date"].max()}')

final_df: (1284440, 8), 2016-01-04 00:00:00 ~ 2026-06-30 00:00:00
sp500_beta_df: (2637, 2), 2016-01-04 00:00:00 ~ 2026-06-30 00:00:00


In [3]:
final_60_df = add_features(final_df, sp500_beta_df=sp500_beta_df, windows=(60,))
final_60_df.to_parquet(PROCESSED_DIR / 'final_60_df.parquet', index=False)
print(f'final_60_df 저장 완료: {final_60_df.shape}, {final_60_df["Date"].min()} ~ {final_60_df["Date"].max()}')
final_60_df.head()

final_60_df 저장 완료: (1284440, 17), 2016-01-04 00:00:00 ~ 2026-06-30 00:00:00


,Date,Ticker,Open,High,Low,Close,Volume,source,volatility_60d,mdd_60d,downside_volatility_60d,beta_60d,ma_gap_60d,rsi_60d,momentum_60d,return_60d,cagr_10y
0,2016-01-04,A,37.751921,37.871444,37.089928,37.411728,3287300,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2016-01-05,A,37.448514,37.650791,37.089936,37.283016,2587200,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2016-01-06,A,36.997993,37.687568,36.823298,37.448513,2103600,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,2016-01-07,A,36.906036,36.915233,35.683192,35.857883,3504300,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,2016-01-08,A,36.060178,36.510699,35.370603,35.480934,3736700,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [4]:
final_120_df = add_features(final_df, sp500_beta_df=sp500_beta_df, windows=(120,))
final_120_df.to_parquet(PROCESSED_DIR / 'final_120_df.parquet', index=False)
print(f'final_120_df 저장 완료: {final_120_df.shape}, {final_120_df["Date"].min()} ~ {final_120_df["Date"].max()}')
final_120_df.head()

final_120_df 저장 완료: (1284440, 17), 2016-01-04 00:00:00 ~ 2026-06-30 00:00:00


,Date,Ticker,Open,High,Low,Close,Volume,source,volatility_120d,mdd_120d,downside_volatility_120d,beta_120d,ma_gap_120d,rsi_120d,momentum_120d,return_120d,cagr_10y
0,2016-01-04,A,37.751921,37.871444,37.089928,37.411728,3287300,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2016-01-05,A,37.448514,37.650791,37.089936,37.283016,2587200,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2016-01-06,A,36.997993,37.687568,36.823298,37.448513,2103600,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,2016-01-07,A,36.906036,36.915233,35.683192,35.857883,3504300,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,2016-01-08,A,36.060178,36.510699,35.370603,35.480934,3736700,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [5]:
cols_120 = ['volatility_120d', 'mdd_120d', 'downside_volatility_120d', 'beta_120d',
            'ma_gap_120d', 'rsi_120d', 'return_120d']
final_120_df[cols_120].corr()

,volatility_120d,mdd_120d,downside_volatility_120d,beta_120d,ma_gap_120d,rsi_120d,return_120d
volatility_120d,1.000000,-0.788601,0.952165,0.521830,-0.012845,-0.150610,0.028307
mdd_120d,-0.788601,1.000000,-0.881787,-0.358578,0.422020,0.565779,0.424743
downside_volatility_120d,0.952165,-0.881787,1.000000,0.475027,-0.214488,-0.354985,-0.195859
beta_120d,0.521830,-0.358578,0.475027,1.000000,0.075805,0.013985,0.137502
ma_gap_120d,-0.012845,0.422020,-0.214488,0.075805,1.000000,0.750737,0.820499
rsi_120d,-0.150610,0.565779,-0.354985,0.013985,0.750737,1.000000,0.806271
return_120d,0.028307,0.424743,-0.195859,0.137502,0.820499,0.806271,1.000000


In [6]:
final_df

,Date,Ticker,Open,High,Low,Close,Volume,source
0,2016-01-04,A,37.751921,37.871444,37.089928,37.411728,3287300,yahoo
1,2016-01-05,A,37.448514,37.650791,37.089936,37.283016,2587200,yahoo
2,2016-01-06,A,36.997993,37.687568,36.823298,37.448513,2103600,yahoo
3,2016-01-07,A,36.906036,36.915233,35.683192,35.857883,3504300,yahoo
4,2016-01-08,A,36.060178,36.510699,35.370603,35.480934,3736700,yahoo
...,...,...,...,...,...,...,...,...
1284435,2026-06-24,ZTS,77.271344,78.542482,76.824463,77.628853,4762100,yahoo
1284436,2026-06-25,ZTS,78.165119,79.356816,76.725160,77.281281,5790300,yahoo
1284437,2026-06-26,ZTS,76.357713,77.072729,75.026993,75.563248,15723300,yahoo
1284438,2026-06-29,ZTS,76.298131,76.357713,72.554225,72.742912,6520300,yahoo


In [7]:
coverage_df

,Ticker,n_rows,actual_start_date,actual_end_date,n_price_imputed,n_ohlc_inconsistent,n_volume_missing,sources,expected_rows_10y,coverage_10y,short_history,has_quality_issue,company,sector
0,A,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,Agilent Technologies,Health Care
1,AAPL,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,Apple Inc.,Information Technology
2,ABBV,2637,2016-01-04,2026-06-30,0,1,0,yahoo,2738,0.9631,False,True,AbbVie,Health Care
3,ABNB,1393,2020-12-10,2026-06-30,0,0,0,yahoo,2738,0.5088,True,False,Airbnb,Consumer Discretionary
4,ABT,2637,2016-01-04,2026-06-30,0,1,0,yahoo,2738,0.9631,False,True,Abbott Laboratories,Health Care
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497,XYZ,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,"Block, Inc.",Financials
498,YUM,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,Yum! Brands,Consumer Discretionary
499,ZBH,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,Zimmer Biomet,Health Care
500,ZBRA,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,Zebra Technologies,Information Technology


In [8]:
final_missing_df

,Ticker,company,sector,fail_stage,fail_reason,n_rows_raw,n_rows_final
0,HONA,Honeywell Aerospace,Industrials,collection,yahoo: rows=11 | chart: None,0,0


In [9]:
sp500_universe

,ticker,company,sector,ticker_original
0,A,Agilent Technologies,Health Care,A
1,AAPL,Apple Inc.,Information Technology,AAPL
2,ABBV,AbbVie,Health Care,ABBV
3,ABNB,Airbnb,Consumer Discretionary,ABNB
4,ABT,Abbott Laboratories,Health Care,ABT
...,...,...,...,...
498,XYZ,"Block, Inc.",Financials,XYZ
499,YUM,Yum! Brands,Consumer Discretionary,YUM
500,ZBH,Zimmer Biomet,Health Care,ZBH
501,ZBRA,Zebra Technologies,Information Technology,ZBRA


In [10]:
final_60_df[['volatility_60d','mdd_60d','downside_volatility_60d','beta_60d']].corr()

,volatility_60d,mdd_60d,downside_volatility_60d,beta_60d
volatility_60d,1.000000,-0.773017,0.930098,0.464279
mdd_60d,-0.773017,1.000000,-0.891505,-0.317425
downside_volatility_60d,0.930098,-0.891505,1.000000,0.409841
beta_60d,0.464279,-0.317425,0.409841,1.000000


In [11]:
final_60_df[['ma_gap_60d', 'rsi_60d', 'momentum_60d', 'return_60d']].corr()

,ma_gap_60d,rsi_60d,momentum_60d,return_60d
ma_gap_60d,1.000000,0.746114,0.834030,0.834030
rsi_60d,0.746114,1.000000,0.848587,0.848587
momentum_60d,0.834030,0.848587,1.000000,1.000000
return_60d,0.834030,0.848587,1.000000,1.000000


In [12]:
final_60_df[['volatility_60d','mdd_60d','downside_volatility_60d','beta_60d','ma_gap_60d', 'rsi_60d', 'momentum_60d', 'return_60d']].corr()

,volatility_60d,mdd_60d,downside_volatility_60d,beta_60d,ma_gap_60d,rsi_60d,momentum_60d,return_60d
volatility_60d,1.000000,-0.773017,0.930098,0.464279,-0.028879,-0.138016,-0.022314,-0.022314
mdd_60d,-0.773017,1.000000,-0.891505,-0.317425,0.471396,0.577435,0.517075,0.517075
downside_volatility_60d,0.930098,-0.891505,1.000000,0.409841,-0.280825,-0.391972,-0.312918,-0.312918
beta_60d,0.464279,-0.317425,0.409841,1.000000,0.048386,-0.003491,0.089547,0.089547
ma_gap_60d,-0.028879,0.471396,-0.280825,0.048386,1.000000,0.746114,0.834030,0.834030
rsi_60d,-0.138016,0.577435,-0.391972,-0.003491,0.746114,1.000000,0.848587,0.848587
momentum_60d,-0.022314,0.517075,-0.312918,0.089547,0.834030,0.848587,1.000000,1.000000
return_60d,-0.022314,0.517075,-0.312918,0.089547,0.834030,0.848587,1.000000,1.000000


In [13]:
final_120_df.columns

Index(['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'source',
       'volatility_120d', 'mdd_120d', 'downside_volatility_120d', 'beta_120d',
       'ma_gap_120d', 'rsi_120d', 'momentum_120d', 'return_120d', 'cagr_10y'],
      dtype='str')

In [14]:
final_120_df[['volatility_120d', 'mdd_120d', 'downside_volatility_120d', 'beta_120d',
       'ma_gap_120d', 'rsi_120d', 'momentum_120d', 'return_120d', 'cagr_10y']].corr()

,volatility_120d,mdd_120d,downside_volatility_120d,beta_120d,ma_gap_120d,rsi_120d,momentum_120d,return_120d,cagr_10y
volatility_120d,1.000000,-0.788601,0.952165,0.521830,-0.012845,-0.150610,0.028307,0.028307,0.296473
mdd_120d,-0.788601,1.000000,-0.881787,-0.358578,0.422020,0.565779,0.424743,0.424743,0.004854
downside_volatility_120d,0.952165,-0.881787,1.000000,0.475027,-0.214488,-0.354985,-0.195859,-0.195859,0.207970
beta_120d,0.521830,-0.358578,0.475027,1.000000,0.075805,0.013985,0.137502,0.137502,0.464319
ma_gap_120d,-0.012845,0.422020,-0.214488,0.075805,1.000000,0.750737,0.820499,0.820499,0.344038
rsi_120d,-0.150610,0.565779,-0.354985,0.013985,0.750737,1.000000,0.806271,0.806271,0.221113
momentum_120d,0.028307,0.424743,-0.195859,0.137502,0.820499,0.806271,1.000000,1.000000,0.358810
return_120d,0.028307,0.424743,-0.195859,0.137502,0.820499,0.806271,1.000000,1.000000,0.358810
cagr_10y,0.296473,0.004854,0.207970,0.464319,0.344038,0.221113,0.358810,0.358810,1.000000


In [15]:
cluster_60df = final_60_df[['Date', 'Ticker', 'beta_60d', 'volatility_60d', 'return_60d', 'rsi_60d']].copy()
print(cluster_60df.shape)
cluster_60df.head()

(1284440, 6)


,Date,Ticker,beta_60d,volatility_60d,return_60d,rsi_60d
0,2016-01-04,A,NaN,NaN,NaN,0.0
1,2016-01-05,A,NaN,NaN,NaN,0.0
2,2016-01-06,A,NaN,NaN,NaN,0.0
3,2016-01-07,A,NaN,NaN,NaN,0.0
4,2016-01-08,A,NaN,NaN,NaN,0.0


In [16]:
cluster_60df

,Date,Ticker,beta_60d,volatility_60d,return_60d,rsi_60d
0,2016-01-04,A,NaN,NaN,NaN,0.000000
1,2016-01-05,A,NaN,NaN,NaN,0.000000
2,2016-01-06,A,NaN,NaN,NaN,0.000000
3,2016-01-07,A,NaN,NaN,NaN,0.000000
4,2016-01-08,A,NaN,NaN,NaN,0.000000
...,...,...,...,...,...,...
1284435,2026-06-24,ZTS,0.658537,0.549101,-0.307366,35.892623
1284436,2026-06-25,ZTS,0.702652,0.545111,-0.328131,34.268887
1284437,2026-06-26,ZTS,0.689590,0.544133,-0.353515,32.673021
1284438,2026-06-29,ZTS,0.614026,0.547763,-0.372763,31.849859


In [17]:
cluster_120df = final_120_df[['Date', 'Ticker', 'beta_120d', 'volatility_120d', 'return_120d', 'rsi_120d']].copy()
print(cluster_120df.shape)


(1284440, 6)


In [18]:
cluster_120df

,Date,Ticker,beta_120d,volatility_120d,return_120d,rsi_120d
0,2016-01-04,A,NaN,NaN,NaN,0.000000
1,2016-01-05,A,NaN,NaN,NaN,0.000000
2,2016-01-06,A,NaN,NaN,NaN,0.000000
3,2016-01-07,A,NaN,NaN,NaN,0.000000
4,2016-01-08,A,NaN,NaN,NaN,0.000000
...,...,...,...,...,...,...
1284435,2026-06-24,ZTS,0.704060,0.427708,-0.376274,38.757980
1284436,2026-06-25,ZTS,0.707814,0.427707,-0.376155,38.761887
1284437,2026-06-26,ZTS,0.709400,0.428515,-0.390508,38.284444
1284438,2026-06-29,ZTS,0.645953,0.428825,-0.428904,36.656747


In [19]:
cluster_60df.to_parquet(PROCESSED_DIR / 'cluster_60df.parquet', index=False)
cluster_120df.to_parquet(PROCESSED_DIR / 'cluster_120df.parquet', index=False)
print(f'cluster_60df 저장 완료: {PROCESSED_DIR / "cluster_60df.parquet"}')
print(f'cluster_120df 저장 완료: {PROCESSED_DIR / "cluster_120df.parquet"}')

cluster_60df 저장 완료: C:\workspaces\lab_middle_project\data\processed\cluster_60df.parquet
cluster_120df 저장 완료: C:\workspaces\lab_middle_project\data\processed\cluster_120df.parquet


# 리밸런싱이 왜 필요한가

특정 날짜 하나만 골라서 군집화하면, "그 순간 안정적인 종목"을 딱 한 번 뽑는 셈이라 **표본이 1개**밖에 안 된다. 그 선택이 좋은 성과로 이어져도, 전략이 진짜 좋은 건지 그냥 운이 좋았던 건지 구분할 수 없다.

그래서 60일마다 시점을 바꿔가며 반복적으로 다시 판단(=리밸런싱)한다. 10년치 데이터를 60일 간격으로 나누면 약 43번의 독립적인 판단 시점이 생기고, 이 43번의 결과를 모아봐야 "이 전략을 꾸준히 실행했을 때 실제로 통했는가"를 통계적으로 의미 있게 검증할 수 있다.

In [20]:
# 매일 다 보면 표본이 너무 많고 서로 겹쳐서(어제와 오늘은 거의 같은 값) 의미가 약하다.
# 60일 간격으로 끊어서, 서로 겹치지 않는 독립적인 판단 시점 43개를 뽑는다.
all_dates = sorted(cluster_60df['Date'].unique())
rebalance_dates = all_dates[::60]
print(f'전체 거래일: {len(all_dates)}, 리밸런싱 시점: {len(rebalance_dates)}개')
print(f'첫 시점: {rebalance_dates[0]}, 마지막 시점: {rebalance_dates[-1]}')

# 리밸런싱 시점에 해당하는 행만 남긴다 (= 매 60일마다 딱 한 번씩 포트폴리오를 다시 짜는 시뮬레이션)
rebalance_snapshots = cluster_60df[cluster_60df['Date'].isin(rebalance_dates)].copy()
print(f'스냅샷 shape: {rebalance_snapshots.shape}')
rebalance_snapshots.head()

전체 거래일: 2637, 리밸런싱 시점: 44개
첫 시점: 2016-01-04 00:00:00, 마지막 시점: 2026-04-09 00:00:00
스냅샷 shape: (21412, 6)


,Date,Ticker,beta_60d,volatility_60d,return_60d,rsi_60d
0,2016-01-04,A,NaN,NaN,NaN,0.000000
60,2016-03-31,A,1.269369,0.285565,-0.020644,49.314042
120,2016-06-24,A,1.231997,0.203317,0.110356,60.183123
180,2016-09-20,A,1.559269,0.226525,0.038768,53.663636
240,2016-12-14,A,1.340917,0.228246,0.011914,51.464841


# 스냅샷이 다음 리밸런싱까지 얼마나 어긋나는지 측정

리밸런싱 시점에 찍은 값(예: `beta_60d`)이 다음 리밸런싱 시점(60일 뒤)엔 얼마나 달라져 있는지, 종목별로 직접 비교한다. 차이가 크면 "스냅샷 하나로 60일을 대표한다"는 가정이 위험하다는 뜻이고, 작으면 비교적 안전하다는 뜻이다.

In [21]:
feature_cols = ['beta_60d', 'volatility_60d', 'return_60d', 'rsi_60d']

# 종목별로 시간순 정렬한 뒤, 바로 이전 리밸런싱 값과의 차이를 구한다 (diff = 이번 값 - 직전 값)
drift = rebalance_snapshots.sort_values(['Ticker', 'Date']).copy()
for col in feature_cols:
    drift[f'{col}_drift'] = drift.groupby('Ticker')[col].diff()

drift_cols = [f'{col}_drift' for col in feature_cols]
drift[drift_cols].describe()

,beta_60d_drift,volatility_60d_drift,return_60d_drift,rsi_60d_drift
count,20410.000000,20410.000000,20410.000000,20911.000000
mean,-0.006758,0.000516,-0.001014,1.222039
std,0.474099,0.158875,0.215938,13.909215
min,-4.115006,-1.712445,-2.675447,-91.133832
25%,-0.259121,-0.059351,-0.117737,-7.622007
50%,-0.007910,-0.002789,-0.004453,0.313871
75%,0.245451,0.050398,0.108964,8.234943
max,4.396111,1.983367,2.468992,70.541454


In [22]:
# drift의 표준편차를 원본 값의 표준편차로 나눠서, "원래 변동폭 대비 얼마나 크게 어긋나는지" 비율로 비교
for col in feature_cols:
    original_std = rebalance_snapshots[col].std()
    drift_std = drift[f'{col}_drift'].std()
    print(f'{col}: 원본 표준편차={original_std:.4f}, drift 표준편차={drift_std:.4f}, 비율={drift_std/original_std:.2f}')

beta_60d: 원본 표준편차=0.5646, drift 표준편차=0.4741, 비율=0.84
volatility_60d: 원본 표준편차=0.1609, drift 표준편차=0.1589, 비율=0.99
return_60d: 원본 표준편차=0.1540, drift 표준편차=0.2159, 비율=1.40
rsi_60d: 원본 표준편차=11.2523, drift 표준편차=13.9092, 비율=1.24


# 알려진 한계: 지표 drift (2026-08-26 결정)

`return_60d`(비율 1.40), `rsi_60d`(비율 1.24)는 리밸런싱 주기(60일) 안에서도 종목 간 차이보다 더 크게 흔들릴 수 있다는 게 위에서 확인됨. `beta_60d`(0.84), `volatility_60d`(0.99)는 상대적으로 덜 흔들림.

**결정**: 지금 구조(지표별 리밸런싱 주기 통일, 60일)를 그대로 유지하고 43번 백테스팅을 먼저 끝까지 돌린다. 지표마다 주기를 다르게 하는 등의 구조 변경은 하지 않는다 — 아직 실제로 성과에 문제가 되는지 확인되지 않은 상태에서 구조를 복잡하게 만들면, 나중에 결과가 나빠졌을 때 원인이 drift 때문인지 다른 요인 때문인지 구분하기 어려워지기 때문이다.

**나중에 다시 볼 것**: 백테스팅 결과가 기대에 못 미치면, 이 drift(특히 `return_60d`/`rsi_60d`)를 원인 후보로 먼저 검토한다.

# 120일 기준 리밸런싱

`cluster_120df`는 지표 자체가 120일 롤링으로 계산돼 있으니, 리밸런싱 간격도 지표 윈도우와 맞춰서 120일로 맞춘다 (60일 리밸런싱과 동일한 로직, 간격만 다름).

In [23]:
all_dates_120 = sorted(cluster_120df['Date'].unique())
rebalance_dates_120 = all_dates_120[::120]
print(f'전체 거래일: {len(all_dates_120)}, 리밸런싱 시점: {len(rebalance_dates_120)}개')
print(f'첫 시점: {rebalance_dates_120[0]}, 마지막 시점: {rebalance_dates_120[-1]}')

rebalance_snapshots_120 = cluster_120df[cluster_120df['Date'].isin(rebalance_dates_120)].copy()
print(f'스냅샷 shape: {rebalance_snapshots_120.shape}')
rebalance_snapshots_120.head()

전체 거래일: 2637, 리밸런싱 시점: 22개
첫 시점: 2016-01-04 00:00:00, 마지막 시점: 2026-01-12 00:00:00
스냅샷 shape: (10697, 6)


,Date,Ticker,beta_120d,volatility_120d,return_120d,rsi_120d
0,2016-01-04,A,NaN,NaN,NaN,0.000000
120,2016-06-24,A,1.253426,0.247355,0.087434,53.613910
240,2016-12-14,A,1.468715,0.226456,0.051144,52.552882
360,2017-06-08,A,1.474503,0.176876,0.310660,64.156386
480,2017-11-28,A,1.185834,0.149368,0.178603,60.050843


In [24]:
feature_cols_120 = ['beta_120d', 'volatility_120d', 'return_120d', 'rsi_120d']

drift_120 = rebalance_snapshots_120.sort_values(['Ticker', 'Date']).copy()
for col in feature_cols_120:
    drift_120[f'{col}_drift'] = drift_120.groupby('Ticker')[col].diff()

for col in feature_cols_120:
    original_std = rebalance_snapshots_120[col].std()
    drift_std = drift_120[f'{col}_drift'].std()
    print(f'{col}: 원본 표준편차={original_std:.4f}, drift 표준편차={drift_std:.4f}, 비율={drift_std/original_std:.2f}')

beta_120d: 원본 표준편차=0.4944, drift 표준편차=0.3495, 비율=0.71
volatility_120d: 원본 표준편차=0.1447, drift 표준편차=0.1303, 비율=0.90
return_120d: 원본 표준편차=0.2533, drift 표준편차=0.3311, 비율=1.31
rsi_120d: 원본 표준편차=12.4847, drift 표준편차=13.8841, 비율=1.11


In [25]:
rebalance_120df=rebalance_snapshots_120.copy()
rebalance_120df

,Date,Ticker,beta_120d,volatility_120d,return_120d,rsi_120d
0,2016-01-04,A,NaN,NaN,NaN,0.000000
120,2016-06-24,A,1.253426,0.247355,0.087434,53.613910
240,2016-12-14,A,1.468715,0.226456,0.051144,52.552882
360,2017-06-08,A,1.474503,0.176876,0.310660,64.156386
480,2017-11-28,A,1.185834,0.149368,0.178603,60.050843
...,...,...,...,...,...,...
1283843,2024-02-12,ZTS,1.047528,0.231560,0.092904,53.860913
1283963,2024-08-05,ZTS,0.835364,0.283740,-0.106536,46.879469
1284083,2025-01-28,ZTS,0.312297,0.218237,-0.019905,49.636948
1284203,2025-07-22,ZTS,0.611297,0.300273,-0.106869,47.243319


In [26]:
rebalance_60df=rebalance_snapshots.copy()

In [27]:
rebalance_60df

,Date,Ticker,beta_60d,volatility_60d,return_60d,rsi_60d
0,2016-01-04,A,NaN,NaN,NaN,0.000000
60,2016-03-31,A,1.269369,0.285565,-0.020644,49.314042
120,2016-06-24,A,1.231997,0.203317,0.110356,60.183123
180,2016-09-20,A,1.559269,0.226525,0.038768,53.663636
240,2016-12-14,A,1.340917,0.228246,0.011914,51.464841
...,...,...,...,...,...,...
1284143,2025-04-24,ZTS,0.558963,0.326089,-0.103310,44.735100
1284203,2025-07-22,ZTS,1.019469,0.274147,-0.003969,50.326528
1284263,2025-10-15,ZTS,0.587705,0.191146,-0.059991,44.940968
1284323,2026-01-12,ZTS,1.138691,0.365011,-0.115140,42.967142


In [28]:
rebalance_60df.to_parquet(PROCESSED_DIR / 'rebalance_60df.parquet', index=False)
rebalance_120df.to_parquet(PROCESSED_DIR / 'rebalance_120df.parquet', index=False)
print(f'rebalance_60df 저장 완료: {PROCESSED_DIR / "rebalance_60df.parquet"}, shape={rebalance_60df.shape}')
print(f'rebalance_120df 저장 완료: {PROCESSED_DIR / "rebalance_120df.parquet"}, shape={rebalance_120df.shape}')

rebalance_60df 저장 완료: C:\workspaces\lab_middle_project\data\processed\rebalance_60df.parquet, shape=(21412, 6)
rebalance_120df 저장 완료: C:\workspaces\lab_middle_project\data\processed\rebalance_120df.parquet, shape=(10697, 6)
